# Comparative Analysis & Advanced Topics
### Lesson 3, Section 3

In this notebook we will:
1. Implement the **SEIR** model and compare it with SIR
2. Run the **agent-based SIR** model alongside the **ODE SIR** model
3. Quantify **stochastic variability** across ABM runs
4. Study the **effect of population size** on convergence to the ODE
5. Implement and test **vaccination strategies**
6. Explore the **herd immunity threshold** numerically

---

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

C_S = '#2ecc71'   # green
C_I = '#e74c3c'   # red
C_R = '#3498db'   # blue
C_E = '#f39c12'   # orange

print('Imports loaded successfully')

---
## 1 · The SEIR Model

The SEIR model adds an **Exposed** compartment to SIR:

$$\frac{dS}{dt} = -\beta S I$$
$$\frac{dE}{dt} = \beta S I - \sigma E$$
$$\frac{dI}{dt} = \sigma E - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

where:
- $\beta$ = infection rate
- $\sigma$ = rate at which exposed become infectious ($1/\sigma$ = latent period)
- $\gamma$ = recovery rate ($1/\gamma$ = infectious period)
- $R_0 = \beta / \gamma$ (same as SIR)

### Step 1: SEIR right-hand side

In [ ]:
def seir_rhs(S, E, I, R, beta, sigma, gamma):
    """Compute rates of change for the SEIR model."""
    dS = -beta * S * I
    dE = beta * S * I - sigma * E
    dI = sigma * E - gamma * I
    dR = gamma * I
    return dS, dE, dI, dR

# Test: no infected and no exposed means no dynamics
test = seir_rhs(1.0, 0.0, 0.0, 0.0, 0.5, 0.2, 0.1)
assert all(v == 0.0 for v in test)
print('Test passed: no infection means no dynamics.')

### Step 2: SEIR Euler simulation

In [ ]:
def simulate_seir(S0, E0, I0, R0, beta, sigma, gamma, T, dt=0.1):
    """Simulate the SEIR model using Euler's method."""
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S, E, I, R = [np.zeros(steps) for _ in range(4)]
    S[0], E[0], I[0], R[0] = S0, E0, I0, R0
    
    for k in range(1, steps):
        dS, dE, dI, dR = seir_rhs(S[k-1], E[k-1], I[k-1], R[k-1],
                                   beta, sigma, gamma)
        S[k] = S[k-1] + dt * dS
        E[k] = E[k-1] + dt * dE
        I[k] = I[k-1] + dt * dI
        R[k] = R[k-1] + dt * dR
    
    return t, S, E, I, R

# Verify conservation
t, S, E, I, R = simulate_seir(0.99, 0.0, 0.01, 0.0, 0.5, 0.2, 0.1, 200)
total = S + E + I + R
assert np.allclose(total, 1.0)
print(f'Conservation check passed. Total population = {total[0]:.4f} throughout.')

In [ ]:
# Plot SEIR dynamics
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, S, color=C_S, lw=2, label='Susceptible')
ax.plot(t, E, color=C_E, lw=2, label='Exposed')
ax.plot(t, I, color=C_I, lw=2, label='Infected')
ax.plot(t, R, color=C_R, lw=2, label='Recovered')
ax.set_xlabel('Time')
ax.set_ylabel('Population fraction')
ax.set_title(f'SEIR Model ($R_0 = {0.5/0.1:.1f}$, latent period = {1/0.2:.0f} days)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Peak infected fraction: {np.max(I):.3f} at t = {t[np.argmax(I)]:.1f}')

### Test: Compare SIR and SEIR

When $\sigma$ is very large (instant transition E → I), the SEIR model should behave like SIR.

In [ ]:
def simulate_sir(S0, I0, R0, beta, gamma, T, dt=0.1):
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S, I, R = np.zeros(steps), np.zeros(steps), np.zeros(steps)
    S[0], I[0], R[0] = S0, I0, R0
    for k in range(1, steps):
        dS = -beta * S[k-1] * I[k-1]
        dI = beta * S[k-1] * I[k-1] - gamma * I[k-1]
        dR = gamma * I[k-1]
        S[k] = S[k-1] + dt * dS
        I[k] = I[k-1] + dt * dI
        R[k] = R[k-1] + dt * dR
    return t, S, I, R

t_sir, S_sir, I_sir, R_sir = simulate_sir(0.99, 0.01, 0.0, 0.5, 0.1, 200)
t_seir_fast, _, _, I_seir_fast, _ = simulate_seir(0.99, 0.0, 0.01, 0.0,
                                                    0.5, 100.0, 0.1, 200)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_sir, I_sir, color=C_I, lw=3, label='SIR')
ax.plot(t_seir_fast, I_seir_fast, color=C_E, lw=2, ls='--', label='SEIR ($\\sigma = 100$)')
ax.set_xlabel('Time')
ax.set_ylabel('Infected fraction')
ax.set_title('SEIR with large $\\sigma$ converges to SIR', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('When sigma is large, SEIR ≈ SIR (E compartment empties instantly).')

---
## 2 · Effect of Latent Period

How does the latent period ($1/\sigma$) affect the epidemic?

In [ ]:
latent_periods = [1, 3, 7, 14]  # days

fig, ax = plt.subplots(figsize=(10, 5))
for lp in latent_periods:
    sigma = 1.0 / lp
    t, _, _, I, _ = simulate_seir(0.99, 0.0, 0.01, 0.0, 0.5, sigma, 0.1, 250)
    ax.plot(t, I, lw=2, label=f'Latent = {lp} days ($\\sigma = {sigma:.2f}$)')

ax.set_xlabel('Time (days)')
ax.set_ylabel('Infected fraction')
ax.set_title('Effect of Latent Period on SEIR Dynamics', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()
print('Longer latent period delays and slightly reduces the peak.')

---
## 3 · Agent-Based SIR Model (for comparison)

We reuse the ABM from Section 2, now with **distance-based transmission**. Instead of requiring agents to share the exact same cell, each infected agent within radius `r_infect` poses an independent risk whose probability decays exponentially with distance:

$$p_i = p_{\text{infect}} \cdot e^{-d / r_{\text{infect}}}$$

The combined probability from multiple nearby infected agents is $P = 1 - \prod_i (1 - p_i)$.

In [ ]:
STATE_S, STATE_I, STATE_R = 1, 2, 3

def update_position(x, y, area, rng):
    d = rng.randint(1, 4)
    if d == 1 and y < area: y += 1
    elif d == 2 and y > 0: y -= 1
    elif d == 3 and x > 0: x -= 1
    elif d == 4 and x < area: x += 1
    return x, y

def compute_infection_probability(idx, positions, states, p_infect, r_infect):
    """Compute infection probability based on proximity to infected agents.
    
    Each infected agent within radius r_infect contributes an independent
    transmission attempt with probability p_infect * exp(-dist / r_infect).
    Returns the combined probability: 1 - prod(1 - p_i).
    """
    if states[idx] != STATE_S:
        return 0.0
    dx = positions[:, 0].astype(float) - positions[idx, 0]
    dy = positions[:, 1].astype(float) - positions[idx, 1]
    dist = np.sqrt(dx**2 + dy**2)
    mask = (states == STATE_I) & (np.arange(len(states)) != idx) & (dist <= r_infect)
    if not np.any(mask):
        return 0.0
    probs = p_infect * np.exp(-dist[mask] / r_infect)
    return 1.0 - np.prod(1.0 - probs)

def simulate_abm_sir(area=40, population=300, initially_infected=5,
                     steps=200, p_infect=0.25, recovery_prob=0.03,
                     r_infect=3.0, vaccinated_fraction=0.0, seed=1):
    """Run agent-based SIR with distance-based transmission."""
    rng = random.Random(seed)
    positions = np.array([[rng.randint(0, area), rng.randint(0, area)]
                          for _ in range(population)], dtype=int)
    states = np.full(population, STATE_S, dtype=int)
    states[:initially_infected] = STATE_I
    
    # Vaccination
    n_vacc = int(vaccinated_fraction * (population - initially_infected))
    vacc_candidates = list(range(initially_infected, population))
    rng.shuffle(vacc_candidates)
    for idx in vacc_candidates[:n_vacc]:
        states[idx] = STATE_R
    
    s_hist, i_hist, r_hist = [], [], []
    for step in range(steps):
        s_hist.append(np.sum(states == STATE_S))
        i_hist.append(np.sum(states == STATE_I))
        r_hist.append(np.sum(states == STATE_R))
        
        new_states = states.copy()
        for p in range(population):
            prob = compute_infection_probability(p, positions, states, p_infect, r_infect)
            if states[p] == STATE_S and prob > 0:
                if rng.random() < prob:
                    new_states[p] = STATE_I
            elif states[p] == STATE_I:
                if rng.random() < recovery_prob:
                    new_states[p] = STATE_R
        states = new_states
        
        for p in range(population):
            x, y = update_position(int(positions[p, 0]), int(positions[p, 1]), area, rng)
            positions[p] = [x, y]
    
    return np.array(s_hist), np.array(i_hist), np.array(r_hist)

s, i, r = simulate_abm_sir(seed=42)
print(f'ABM simulation done. Peak infected: {np.max(i)} agents.')

---
## 4 · ODE vs ABM: Direct Comparison

In [ ]:
pop = 300
steps = 200

# ODE (tuned to roughly match ABM dynamics)
t_ode, S_ode, I_ode, R_ode = simulate_sir(0.99, 0.01, 0.0, 0.30, 0.03, steps, dt=0.5)

# ABM
s_abm, i_abm, r_abm = simulate_abm_sir(population=pop, steps=steps, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: ODE
ax1.plot(t_ode, S_ode, color=C_S, lw=2, label='S')
ax1.plot(t_ode, I_ode, color=C_I, lw=2, label='I')
ax1.plot(t_ode, R_ode, color=C_R, lw=2, label='R')
ax1.set_title('ODE SIR Model (deterministic)', fontweight='bold')
ax1.set_xlabel('Time')
ax1.set_ylabel('Fraction')
ax1.legend()

# Right: ABM
t_abm = np.arange(steps)
ax2.plot(t_abm, s_abm / pop, color=C_S, lw=2, label='S')
ax2.plot(t_abm, i_abm / pop, color=C_I, lw=2, label='I')
ax2.plot(t_abm, r_abm / pop, color=C_R, lw=2, label='R')
ax2.set_title('Agent-Based SIR (stochastic)', fontweight='bold')
ax2.set_xlabel('Time step')
ax2.set_ylabel('Fraction')
ax2.legend()

plt.suptitle('Equation-Based vs Agent-Based SIR', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

---
## 5 · Stochastic Variability

Each ABM run with a different seed produces a different trajectory. Let's quantify this.

In [ ]:
n_runs = 20
all_infected = []

fig, ax = plt.subplots(figsize=(10, 5))
for seed in range(1, n_runs + 1):
    _, i, _ = simulate_abm_sir(population=300, steps=200, seed=seed)
    all_infected.append(i)
    ax.plot(i / 300, color=C_I, lw=0.7, alpha=0.4)

# Mean and std
all_infected = np.array(all_infected)
mean_i = np.mean(all_infected, axis=0)
std_i = np.std(all_infected, axis=0)

ax.plot(mean_i / 300, color='black', lw=3, label='Mean of 20 runs')
ax.fill_between(range(200), (mean_i - std_i) / 300, (mean_i + std_i) / 300,
                alpha=0.2, color='gray', label='$\\pm$ 1 std')

ax.set_xlabel('Time step')
ax.set_ylabel('Infected fraction')
ax.set_title('Stochastic Variability Across 20 ABM Runs', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

peaks = np.max(all_infected, axis=1)
print(f'Peak infected — mean: {np.mean(peaks):.1f}, std: {np.std(peaks):.1f}, '
      f'min: {np.min(peaks)}, max: {np.max(peaks)}')

---
## 6 · Effect of Population Size

As the population grows, the ABM should converge toward the ODE (law of large numbers).

In [ ]:
pop_sizes = [50, 300, 2000]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, N in zip(axes, pop_sizes):
    for seed in range(1, 11):
        _, i, _ = simulate_abm_sir(
            area=int(np.sqrt(N) * 3), population=N,
            initially_infected=max(1, N // 50),
            steps=200, seed=seed
        )
        ax.plot(i / N, color=C_I, lw=0.8, alpha=0.5)
    ax.set_title(f'N = {N}', fontweight='bold')
    ax.set_xlabel('Time step')
    ax.set_ylim(-0.02, 0.6)
axes[0].set_ylabel('Infected fraction')

fig.suptitle('Convergence: Larger Populations → Less Variability', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('With N=2000, the curves cluster tightly — approaching the deterministic ODE limit.')

---
## 7 · Vaccination Strategies

The herd immunity threshold is $p_{\mathrm{herd}} = 1 - 1/R_0$. Let's test this in the ABM.

In [ ]:
vacc_fractions = [0.0, 0.2, 0.4, 0.6, 0.8]

fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(vacc_fractions)))

for vf, c in zip(vacc_fractions, colors):
    _, i, _ = simulate_abm_sir(population=300, steps=200,
                                vaccinated_fraction=vf, seed=42)
    ax.plot(i, color=c, lw=2, label=f'{int(vf*100)}% vaccinated')

ax.set_xlabel('Time step')
ax.set_ylabel('Infected agents')
ax.set_title('Effect of Vaccination Coverage', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8 · Herd Immunity: Mathematical View

The herd immunity threshold is:

$$p_{\mathrm{herd}} = 1 - \frac{1}{R_0}$$

Let's plot this and mark some real diseases.

In [ ]:
R0_range = np.linspace(1.01, 20, 200)
p_herd = 1 - 1 / R0_range

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(R0_range, p_herd * 100, color='#8e44ad', lw=3)
ax.fill_between(R0_range, p_herd * 100, alpha=0.15, color='#8e44ad')

diseases = [('Influenza', 1.3), ('COVID-19', 2.5), ('Smallpox', 5.0),
            ('Mumps', 7.0), ('Measles', 15.0)]
for name, r0 in diseases:
    p = (1 - 1/r0) * 100
    ax.plot(r0, p, 'o', markersize=10, color='#e74c3c', zorder=5)
    ax.annotate(f'{name}\n$R_0={r0}$', xy=(r0, p),
                xytext=(r0 + 0.5, p - 8), fontsize=9)

ax.set_xlabel('Basic Reproduction Number $R_0$')
ax.set_ylabel('Herd Immunity Threshold (%)')
ax.set_title('$p_{herd} = 1 - 1/R_0$', fontweight='bold')
ax.set_xlim(1, 20)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

---
## 9 · What to Try Next

1. **Infection radius sweep**: Compare ABM epidemic curves for `r_infect` ∈ {1, 2, 3, 5}. How does the infection radius interact with `p_infect`?
2. **SEIR with vaccination**: Add a vaccination rate that moves susceptible people directly to R. What coverage prevents an epidemic?
3. **Network effects**: Modify the ABM so agents only interact with a fixed set of "contacts" instead of spatial proximity. How does this change the dynamics?
4. **Quarantine**: Implement a rule where infected agents stop moving after detection (with some delay). How does detection speed affect the outbreak size?
5. **R₀ estimation**: Given an ABM epidemic curve, estimate R₀ from the early exponential growth phase. Compare with the known parameter values.
6. **Final size**: For the ODE SIR model, solve the final size equation $S_\infty = S_0 \exp(-R_0(1 - S_\infty))$ numerically using `scipy.optimize.fsolve`. Compare with the simulation.

---

## 10 · Mini-Projects

### Project A: SEIR Agent-Based Model
Extend the ABM with an Exposed state. Agents transition S → E (on contact) → I (after latent period) → R. Use the distance-based transmission model with `r_infect`. Compare the ABM SEIR curves with the ODE SEIR for different latent periods.

### Project B: Superspreaders
Give 5% of agents a 10× higher infection probability or a larger personal `r_infect`. How does this change the epidemic? Compare the total infected with the homogeneous case.

### Project C: Intervention Timing
Implement "lockdown" (reduce movement by 80%) that starts at different times: t=10, t=30, t=50. You can also model social distancing by reducing `r_infect` during lockdown. Plot the total infected as a function of lockdown start time. When is intervention most effective?

### Project D: Fitting R₀ to Data
Use the English boarding school data (SIR model from Section 1). Try different values of $\beta$ and $\gamma$ to find the best fit. Estimate $R_0$ for that outbreak.

### Project E: Distance vs Density
Run the ABM with a fixed `r_infect` but vary the grid `area` while keeping population constant. This changes agent density. How does crowding interact with the infection radius to shape the epidemic?

### Final Reflection
After completing the projects, discuss:
- When is an ODE model sufficient and when do you need an ABM?
- What are the practical implications of stochastic variability for public health decisions?
- How does the infection radius parameter relate to real-world interventions like social distancing?
- How would you validate an epidemic model against real data?